# 02 — Data Preparation

## Superstore Sales Analytics

Este notebook tem como objetivo executar e validar o pipeline de preparação dos dados definido em src/data_cleaning.py.

### Objetivos

- Carregar o dataset original a partir de data/raw
- Aplicar as regras de limpeza identificadas na auditoria de qualidade
- Validar o resultado das transformações
- Preservar a granularidade da linha de venda
- Gerar um dataset tratado e reproduzível em data/processed

### Regras de tratamento

- Converter Order Date e Ship Date para datetime
- Converter Postal Code para identificador textual
- Preservar os valores ausentes de Postal Code
- Remover duplicidades de negócio desconsiderando Row ID
- Preservar o dataset original sem alterações

> As inconsistências relacionadas à modelagem de produtos e geografia serão tratadas posteriormente na construção das estruturas analíticas, evitando alterações arbitrárias nos dados de origem.

## 1. Configuração e Reprodutibilidade

### 1.1 — Configuração do Ambiente e Caminhos

A preparação inicia pela definição programática da raiz do projeto e dos diretórios de entrada e saída. O uso de caminhos relativos evita dependência de diretórios absolutos da máquina local e melhora a portabilidade do pipeline.

A raiz do projeto também é adicionada ao sys.path para permitir a importação do módulo interno src.data_cleaning, mantendo as regras de transformação desacopladas do notebook.

In [15]:
from pathlib import Path
import sys

import pandas as pd


PROJECT_ROOT = Path.cwd().parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_cleaning import clean_sales_data

## 2. Ingestão dos Dados

### 2.1 — Carregamento da Fonte Bruta

O dataset é carregado diretamente da camada data/raw, que representa a fonte original e imutável do pipeline.

O DataFrame é mantido como df_raw para preservar explicitamente o estado de entrada e garantir rastreabilidade entre os dados de origem e o resultado das transformações.

In [16]:
file_path = DATA_RAW / "sales_data.xlsx"

df_raw = pd.read_excel(file_path)

print("Dataset bruto carregado com sucesso.")
print(f"Linhas: {df_raw.shape[0]:,}")
print(f"Colunas: {df_raw.shape[1]}")

Dataset bruto carregado com sucesso.
Linhas: 9,800
Colunas: 19


## 3. Execução do Pipeline de Transformação

### 3.1 — Aplicação das Regras de Limpeza

As regras de tratamento identificadas durante a auditoria de qualidade são aplicadas por meio de clean_sales_data().

A lógica de transformação permanece centralizada em src/data_cleaning.py, enquanto este notebook atua como camada de orquestração, validação e persistência. O resultado é armazenado em df_clean, mantendo df_raw inalterado.

In [17]:
df_clean = clean_sales_data(df_raw)

print("Pipeline de limpeza executado com sucesso.")
print(f"Linhas antes da limpeza: {len(df_raw):,}")
print(f"Linhas após a limpeza: {len(df_clean):,}")
print(f"Registros removidos: {len(df_raw) - len(df_clean):,}")

Pipeline de limpeza executado com sucesso.
Linhas antes da limpeza: 9,800
Linhas após a limpeza: 9,799
Registros removidos: 1


## 4. Quality Gates Pós-Transformação

### 4.1 — Validação de Schema e Tipos

Após a execução do pipeline, são verificados os tipos das variáveis transformadas. O objetivo é confirmar que as datas passaram a possuir representação temporal e que Postal Code deixou de ser tratado como variável numérica, passando a representar corretamente um identificador textual.

In [18]:
df_clean[
    ["Order Date", "Ship Date", "Postal Code"]
].dtypes

Order Date     datetime64[us]
Ship Date      datetime64[us]
Postal Code            string
dtype: object

### 4.2 — Validação de Completude

A transformação de Postal Code deve preservar os 11 valores ausentes identificados na auditoria inicial.

Este controle verifica que a conversão de tipo não introduziu valores artificiais, não eliminou registros e não converteu ausências em representações textuais como "nan".

In [19]:
missing_postal = df_clean["Postal Code"].isna().sum()

print(f"Postal Codes ausentes após a limpeza: {missing_postal:,}")

Postal Codes ausentes após a limpeza: 11


### 4.3 — Validação Pontual da Duplicidade Identificada

A auditoria identificou os registros 3406 e 3407 como uma duplicidade de negócio, diferenciados exclusivamente pelo identificador técnico Row ID.

A verificação abaixo confirma que a regra keep="first" preservou a primeira ocorrência e removeu a segunda.

In [20]:
df_clean[
    df_clean["Row ID"].isin([3406, 3407])
][
    ["Row ID", "Order ID", "Product ID", "Sales"]
]

,Row ID,Order ID,Product ID,Sales
6623,3406,US-2015-150119,FUR-CH-10002965,281.372


### 4.4 — Validação Global de Unicidade de Negócio

Além da verificação do caso previamente conhecido, todo o dataset processado é submetido novamente à regra de identificação de duplicidades.

Row ID é excluído da comparação por representar um identificador técnico. O controle garante que nenhuma duplicidade de negócio permaneça após a transformação.

In [21]:
business_columns = [
    column for column in df_clean.columns
    if column != "Row ID"
]

remaining_duplicates = df_clean.duplicated(
    subset=business_columns
).sum()

print(
    f"Duplicidades de negócio restantes: "
    f"{remaining_duplicates:,}"
)

Duplicidades de negócio restantes: 0


### 4.5 — Validação de Integridade Temporal

A conversão das variáveis de data não deve alterar a relação temporal existente entre pedido e envio.

São avaliadas duas regras de integridade:

- Ship Date não pode ser anterior a Order Date;
- a diferença entre ambas deve permanecer consistente com Shipping Days.

Esses controles verificam que a transformação preservou a semântica temporal do dataset.

In [22]:
calculated_shipping_days = (
    df_clean["Ship Date"] - df_clean["Order Date"]
).dt.days

shipping_days_inconsistencies = (
    calculated_shipping_days != df_clean["Shipping Days"]
).sum()

invalid_shipping_dates = (
    df_clean["Ship Date"] < df_clean["Order Date"]
).sum()

print(
    f"Inconsistências em Shipping Days: "
    f"{shipping_days_inconsistencies:,}"
)

print(
    f"Envios anteriores ao pedido: "
    f"{invalid_shipping_dates:,}"
)

Inconsistências em Shipping Days: 0
Envios anteriores ao pedido: 0


### 4.6 — Consolidação dos Controles de Qualidade

Os principais indicadores de validação são consolidados em uma única estrutura para registrar o estado final do dataset antes da persistência.

Esta etapa funciona como um checkpoint técnico entre transformação e armazenamento da camada processada.

In [23]:
validation_summary = pd.Series({
    "Linhas processadas": len(df_clean),
    "Colunas": df_clean.shape[1],
    "Postal Codes ausentes": df_clean["Postal Code"].isna().sum(),
    "Duplicidades de negócio": remaining_duplicates,
    "Datas de envio inválidas": invalid_shipping_dates,
    "Inconsistências Shipping Days": shipping_days_inconsistencies,
})

validation_summary

Linhas processadas               9799
Colunas                            19
Postal Codes ausentes              11
Duplicidades de negócio             0
Datas de envio inválidas            0
Inconsistências Shipping Days       0
dtype: int64

### 4.7 — Resultado dos Quality Gates

Após a aplicação do pipeline de limpeza, o dataset passou de 9.800 para 9.799 registros devido à remoção de uma duplicidade de negócio previamente identificada.

As principais validações confirmaram que:

- Order Date e Ship Date foram convertidas para formato temporal;
- Postal Code foi convertido para identificador textual;
- os 11 valores ausentes de Postal Code foram preservados;
- não permanecem duplicidades de negócio;
- não existem datas de envio anteriores às datas dos pedidos;
- Shipping Days permanece consistente com a diferença entre Ship Date e Order Date;
- a estrutura original de 19 colunas foi preservada.

O dataset processado está, portanto, preparado para persistência em data/processed e utilização nas etapas posteriores de análise.

## 5. Persistência da Camada Processada

### 5.1 — Exportação para Parquet

Após a aprovação dos controles de qualidade, o dataset é persistido na camada data/processed.

O formato Parquet foi adotado por preservar o schema de dados, oferecer armazenamento colunar e evitar a necessidade de reconstrução manual dos tipos nas etapas analíticas subsequentes.

O índice interno do pandas não é persistido, pois não representa um atributo de negócio.

In [24]:
output_file = DATA_PROCESSED / "sales_data_clean.parquet"

df_clean.to_parquet(
    output_file,
    engine="pyarrow",
    index=False,
)

print(f"Dataset processado salvo em: {output_file}")

Dataset processado salvo em: c:\superstore-analytics\data\processed\sales_data_clean.parquet


### 5.2 — Validação Pós-Persistência

O arquivo persistido é relido a partir do armazenamento para validar o artefato efetivamente gravado, e não apenas o DataFrame mantido em memória.

São verificadas a quantidade de registros e a quantidade de colunas após o ciclo de escrita e leitura.

In [25]:
df_processed = pd.read_parquet(
    output_file,
    engine="pyarrow",
)

print("Dataset processado recarregado com sucesso.")
print(f"Linhas: {df_processed.shape[0]:,}")
print(f"Colunas: {df_processed.shape[1]}")

Dataset processado recarregado com sucesso.
Linhas: 9,799
Colunas: 19


### 5.3 — Validação do Schema Persistido

A última verificação confirma que os tipos críticos definidos durante a preparação foram preservados pelo formato Parquet após a persistência e releitura.

Este controle encerra o ciclo de validação entre a fonte bruta, a transformação em memória e o artefato processado armazenado.

In [26]:
df_processed[
    ["Order Date", "Ship Date", "Postal Code"]
].dtypes

Order Date     datetime64[us]
Ship Date      datetime64[us]
Postal Code            string
dtype: object

## Conclusão

O pipeline de preparação dos dados foi executado e validado com sucesso.

O dataset original foi preservado em data/raw, enquanto as transformações foram aplicadas de forma reproduzível por meio das funções definidas em src/data_cleaning.py.

Após o tratamento:

- o dataset passou de 9.800 para 9.799 registros devido à remoção de uma duplicidade de negócio;
- as 19 colunas originais foram preservadas;
- Order Date e Ship Date foram convertidas para formato temporal;
- Postal Code foi convertido para identificador textual, preservando os 11 valores ausentes;
- não permaneceram duplicidades de negócio;
- não foram identificadas inconsistências temporais;
- o dataset tratado foi persistido em formato Parquet e validado após sua releitura.

O arquivo data/processed/sales_data_clean.parquet será utilizado como fonte das próximas etapas analíticas, iniciando pela análise exploratória em 03_eda.ipynb.